# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). This schema describes the collection: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and fields (columns). All entities are referenced by their `@id` as per dataset schema.

Let's print the available record set IDs and, for each, the corresponding field (`@id`) definitions, so you can select fields for analysis.

In [ ]:
# List record set IDs and their fields (@id)
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print("Record sets (@id):")
for rs in metadata.to_json().get('recordSet', []):
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    # Show fields
    print("  Fields / Columns:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    - {field.get('@id', '')}: {field.get('name', '')} ({field.get('dataType', '')})")
        else:
            print(f"    - {field}")
    print("")
# If no recordSets present, give a message
if not record_set_ids:
    print("No record sets defined in this schema.")

## 3. Data Extraction
Load data from available record set(s) into a DataFrame for analysis. Use the record set and field `@id`s identified above. 

**Note:** If the schema defined no record sets, we will attempt to access via available distributions, columns, or try default dataset loading.

In [ ]:
# Attempt to extract data from each record set
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records for {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Fields in {record_set_id}: {df.columns.tolist()}")
        dataframes[record_set_id] = df
        print(df.head(), "\n")
else:
    # If no record sets are defined, fallback to generic record loading
    print('No explicit record sets found; attempting to load all records (dataset.records())')
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print(f"Fields found: {df.columns.tolist()}")
        print(df.head())
        dataframes['default'] = df
    except Exception as e:
        print(f'Error loading records: {e}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. 

Below is a generic example on how to filter, normalize, and group by a field. **Update the field IDs as appropriate based on the output from Section 3.**

In [ ]:
import numpy as np

# Example: choose which dataframe and which IDs for a numeric field and group field
if dataframes:
    # Use the first loaded dataframe
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}\nColumns: {df.columns.tolist()}")
    # Guess a numeric field (if any field appears numeric)
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Numeric field selected: {numeric_field}")
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
        # Filter records with numeric_field > threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records (where {numeric_field} > {threshold}):\n", filtered_df.head())
        # Normalize
        filtered_df[f'{numeric_field}_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
        # Try grouping (use the first non-numeric column if any)
        group_fields = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped by {group_field} (showing means):\n{grouped_df.head()}")
        else:
            print("No suitable group fields found for grouping.")
    else:
        print("No numeric fields found.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize the distribution of a selected numeric field. This helps you to quickly check the field's shape and inspect for outliers.

**Adjust `numeric_field` as appropriate.**

In [ ]:
import matplotlib.pyplot as plt

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Attempt to reuse numeric_field from previous cell if defined
    try:
        numeric_col = numeric_field
        plt.figure(figsize=(8,4))
        df[numeric_col].hist(bins=25)
        plt.title(f"Distribution of {numeric_col}")
        plt.xlabel(numeric_col)
        plt.ylabel("Count")
        plt.show()
    except Exception as e:
        print(f"Couldn't plot histogram. Error: {e}")
else:
    print("No data to visualize.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and perform basic processing on the FAIR^2 dataset using the `mlcroissant` library. You can repeat/extend the steps above by selecting specific record sets and field `@id`s that are of interest for your research and downstream analysis.

- Use the `@id` references provided by the schema for all data access.
- Explore and visualize other numeric or categorical fields using the same approach.

### Resources
- [FAIR^2 Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- [`mlcroissant` documentation](https://mlcommons.org/croissant/).
